## Computational Physics -- Problem Set 5
##### _Theodore Landa_

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import os
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

### Problem 1.

_(e)_ Hogg made this problem so that we would figure out how to manipulate files in our code, and in particular, how to make files that rely on each others' contents and automatically update when the dependent file is updated!

In [ ]:
def make_hello():
    if not os.path.exists("hello.txt"):
        with open("hello.txt", "w") as f1:
            f1.write("hello")
        print("I'm making file hello.txt!")

In [ ]:
make_hello()
make_hello()

In [ ]:
def make_goodbye():
    if os.path.exists("hello.txt"):
        if os.path.exists("goodbye.txt"):
            if (os.path.getmtime("hello.txt") > os.path.getmtime("goodbye.txt")):
                os.remove("goodbye.txt")
                make_goodbye()
        else:
            with open("goodbye.txt", "w") as f2, open("hello.txt", "r") as f1:
                f2.write(f1.read() + " goodbye")
            print("I'm making file goodbye.txt!")
    else:
        make_hello()
        make_goodbye()

In [ ]:
make_hello()
make_goodbye()
with open("goodbye.txt", "r") as test:
    print(test.read())
os.remove("hello.txt")
make_hello()
make_goodbye()

### Problem 2.

_(d)_ The plots do agree in the relevant ways! ie they look like reflections of each other across the E=M line. I plotted a finer grid version as well as the 12 angle version, so that this would be particularly apparent.

In [ ]:
def M_from_E(E, e):
    return E - e*jnp.sin(E)

In [ ]:
def dM_dE(E, e):
    return jax.grad(M_from_E)(E, e)

def E_from_M(M, e):
    E = M
    compare = M - 1
    tol = 1e-6
    while (np.abs(E - compare) > tol):
        compare = E
        E = E - (M_from_E(E, e) - M) / dM_dE(E, e)
    return E

In [ ]:
def plotME(grid):
    Es = np.linspace(0, 2*np.pi, grid)
    es = np.array([0., 0.5, 0.75, 1.5])
    Ms0 = M_from_E(Es, es[0])
    Ms05 = M_from_E(Es, es[1])
    Ms075 = M_from_E(Es, es[2])
    Ms095 = M_from_E(Es, es[3])
    plt.scatter(Es, Ms0, color = "k", label = "e = 0.")
    plt.scatter(Es, Ms05, color = "r", label = "e = 0.5")
    plt.scatter(Es, Ms075, color = "g", label = "e = 0.75")
    plt.scatter(Es, Ms095, color = "b", label = "e = 0.95")
    plt.legend()
    plt.xlabel("E (radians)")
    plt.ylabel("M (radians)")
    plt.title("M(E) with four eccentricities")

def plotEM(grid):
    es = np.array([0., 0.5, 0.75, 1.5])
    Ms = np.linspace(0, 2*np.pi, grid)
    Es0 = np.zeros(grid)
    Es05 = np.zeros(grid)
    Es075 = np.zeros(grid)
    Es095 = np.zeros(grid)
    for j in range(grid):
        Es0[j] = E_from_M(Ms[j], es[0])
        Es05[j] = E_from_M(Ms[j], es[1])
        Es075[j] = E_from_M(Ms[j], es[2])
        Es095[j] = E_from_M(Ms[j], es[3])
    plt.scatter(Ms, Es0, color = "k", label = "e = 0.")
    plt.scatter(Ms, Es05, color = "r", label = "e = 0.5")
    plt.scatter(Ms, Es075, color = "g", label = "e = 0.75")
    plt.scatter(Ms, Es095, color = "b", label = "e = 0.95")
    plt.scatter(M_from_E(Ms, es[3]), Ms, color="b", marker="x")
    plt.legend()
    plt.xlabel("M (radians)")
    plt.ylabel("E (radians)")
    plt.title("E(M) with four eccentricities")

In [ ]:
plotME(24)

In [ ]:
plotEM(24)

In [ ]:
plotME(100)

In [ ]:
plotEM(100)

### Problem 3.

_(a)_ What goes in must come out. The input current to $V_2$ is $I_2+I_D$, given by $$(V^{+}-V_2)=I_2\,R_2$$ and $$I_D=I_0\,(e^{(V_1-V_2)/V_T}-1)$$ The output current to $V_2$ is $I_4$, given by $$V_2=I_4\,R_4$$ Setting $I_2+I_D=I_4$, we get $$\frac{(V^{+}-V_2)}{R_2}-\frac{V_2}{R_4}+I_0\,(e^{(V_1-V_2)/V_T}-1)=0$$

_(b)_ My voltages are V1 = `3.222567324942014`V and V2 = `2.6043247733903554`V.

_(c)_ The voltage across the diode is `0.6182425515516585`V. My results agree with the electronic engineer's rule of thumb!

In [ ]:
Vplus = 5.
R1 = 1e3
R2 = 4*1e3
R3 = 3*1e3
R4 = 2*1e3
I0 = 3*1e-9
VT = 0.05

In [ ]:
def f(V):
    V1 = V[0]
    V2 = V[1]
    return jnp.asarray([((V1 - Vplus) / R1) + (V1 / R3) + I0 * (jnp.exp((V1 - V2) / VT) - 1.),
                       ((Vplus - V2) / R2) - (V2 / R4) + I0 * (jnp.exp((V1 - V2) / VT) - 1.)])

In [ ]:
def jac(V):
    return jax.jacrev(f)(V)

def voltages():
    V = np.array([0.9, 0.3])
    compare = np.array([0., 0.])
    tol = 1e-7
    while (np.abs(V - compare) > tol).any():
        compare = V
        V = V - np.linalg.solve(jac(V), f(V))
    return V

In [ ]:
V1, V2 = voltages()
print(V1, V2)
print(V1-V2)

### Problem 4.

_(a)_ Our dimensionless (k = m = 1, so we measure frequencies relative to $\omega_0$) frequencies are `[2.61312593, 2.32684627, 2.32684627, 2., 2., 2., 1.84775907, 1.84775907, 1.60803807, 1.60803807, 1.41421356, 1.41421356, 1.0823922, 0.76536686, 0.76536686]`. Note that there are degeneracies: several eigenvalues have the same frequency.

_(b)_ The lowest possible frequency is `0.76536686`, which corresponds to an antisymmetric vibrational mode: the lattice acts like a seesaw across a diagonal.

_(c)_ Plot!

_(d)_ Plot!

_(e)_ Plot!

In [ ]:
def make_laplacian(n):
    L = np.zeros([n**2, n**2])
    for i in range(n):
        for j in range(n):
            if (i == 0):
                if (j == 0):
                    L[i+j*n][i+j*n+1] = 1
                    L[i+j*n][i+j*n+n] = 1
                    L[i+j*n][i+j*n] = -2
                elif (j == n-1):
                    L[i+j*n][i+j*n+1] = 1
                    L[i+j*n][i+j*n-n] = 1
                    L[i+j*n][i+j*n] = -2
                else:
                    L[i+j*n][i+j*n+1] = 1
                    L[i+j*n][i+j*n+n] = 1
                    L[i+j*n][i+j*n-n] = 1
                    L[i+j*n][i+j*n] = -3
            elif (i == n-1):
                if (j == 0):
                    L[i+j*n][i+j*n-1] = 1
                    L[i+j*n][i+j*n+n] = 1
                    L[i+j*n][i+j*n] = -2
                elif (j == n-1):
                    L[i+j*n][i+j*n-1] = 1
                    L[i+j*n][i+j*n-n] = 1
                    L[i+j*n][i+j*n] = -2
                else:
                    L[i+j*n][i+j*n-1] = 1
                    L[i+j*n][i+j*n+n] = 1
                    L[i+j*n][i+j*n-n] = 1
                    L[i+j*n][i+j*n] = -3
            elif (j == 0):
                L[i+j*n][i+j*n-1] = 1
                L[i+j*n][i+j*n+n] = 1
                L[i+j*n][i+j*n+1] = 1
                L[i+j*n][i+j*n] = -3
            elif (j == n-1):
                L[i+j*n][i+j*n-1] = 1
                L[i+j*n][i+j*n-n] = 1
                L[i+j*n][i+j*n+1] = 1
                L[i+j*n][i+j*n] = -3
            else:
                L[i+j*n][i+j*n-1] = 1
                L[i+j*n][i+j*n+1] = 1
                L[i+j*n][i+j*n-n] = 1
                L[i+j*n][i+j*n+n] = 1
                L[i+j*n][i+j*n] = -4
    return L

In [ ]:
L4 = make_laplacian(4)
print(L4)

In [ ]:
eigvals4, eigvecs4 = np.linalg.eigh(L4)
print(np.sqrt(np.round(-1 * eigvals4, 14)))

In [ ]:
eigarr4 = np.zeros([4, 4])
for u in range(16):
    eigarr4[u % 4][np.int32(u / 4)] = eigvecs4[u][-2]
plt.imshow(eigarr4)
plt.xticks(range(4))
plt.yticks(range(4))
plt.title("Lowest Frequency Mode!")
plt.gray()

In [ ]:
L50 = make_laplacian(70)
eigvals50, eigvecs50 = np.linalg.eigh(L50)

In [ ]:
print(np.sqrt(np.round(-1 * eigvals50, 14)))
eigarr50 = np.zeros([70, 70])
for u in range(70**2):
    eigarr50[u % 70][np.int32(u / 70)] = eigvecs50[u][-85]
plt.imshow(eigarr50, cmap="RdBu", vmin=-2. / 70, vmax=2. / 70)
plt.axis("off")
plt.title("Lowest Frequency Mode!")
plt.show()

In [ ]:
#Time to ball.
fig, ax = plt.subplots(5, 5, figsize=(10, 10))
for i in range(5):
    for j in range(5):
        eigarr = np.zeros([50, 50])
        for u in range(50**2):
            eigarr[u % 50][np.int32(u / 50)] = eigvecs50[u][-(i+j*5)-2]
        ax[i][j].imshow(eigarr, cmap="viridis")
        ax[i][j].axis("off")
        ax[i, j].set_aspect("equal")
plt.tight_layout()

### Credits.
The numpy documentation, matplotlib documentation, jax documentation, Seth Murchison for discussion of good Newton's method guesses, Gabe Rosen-Turits for confirming an answer, and geeksforgeeks for a great explanaion of python with for opening text files.